In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diabetes-dataset/train_dev_test_data.pkl


## In this notebook, the same model for diabetes prediction will be implmented using PyTorch. 

In [2]:
import pickle 
import torch
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim 
import torch.nn.functional as F 

#### Loading Previously split dataset

# Importing processed split dataset from previous notebook "Feature Engineering"
with open ("/kaggle/input/diabetes-dataset/train_dev_test_data.pkl", "rb") as f:
    X_train, X_dev, X_test, y_train, y_dev, y_test = pickle.load(f)

#### Converting datasets to tensors then to tensorDatasets and finally establish the dataloader

# Convert pandas DataFrames to tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_dev_tensor = torch.tensor(X_dev.values, dtype=torch.float32)
y_dev_tensor = torch.tensor(y_dev.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
dev_dataset = TensorDataset(X_dev_tensor, y_dev_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

# Sanity Check 
print (X_train_tensor.shape)
print (y_train_tensor.shape)

# Model Architecture # Layers and # Hidden Units 
# Manual - Non dynamic model
layer_dims = [X_train_tensor.shape[1], 5, 1] 
model = nn.Sequential (
                        nn.Linear(X_train_tensor.shape[1], 6),
                        nn.ReLU(),
                        nn.Linear(6, 6),
                        nn.ReLU(),
                        nn.Linear(6,5),
                        nn.ReLU(),
                        nn.Linear(5,1),
)

# dynamic Model for any layer_dims 
layers = []
for i in range (len(layer_dims) - 2): 
    layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
    layers.append(nn.ReLU())

layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))
model = nn.Sequential(*layers)

print (model)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss_epoch = 0
    correct_train = 0
    total_train = 0
    
    for X_batch, y_batch in train_loader:
        y_batch = y_batch.view(-1, 1).float()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss_epoch += loss.item() * X_batch.size(0)
        
        # Compute number of correct predictions in this batch
        preds = torch.sigmoid(outputs) >= 0.5
        correct_train += (preds.float() == y_batch).sum().item()
        total_train += y_batch.size(0)
    
    train_loss_epoch /= len(train_loader.dataset)
    train_acc = correct_train / total_train
    
    # Evaluate dev set
    model.eval()
    with torch.no_grad():
        dev_outputs = model(X_dev_tensor)
        dev_loss = criterion(dev_outputs, y_dev_tensor.view(-1,1).float())
        
        preds_dev = torch.sigmoid(dev_outputs) >= 0.5
        correct_dev = (preds_dev.float() == y_dev_tensor.view(-1,1).float()).sum().item()
        dev_acc = correct_dev / len(y_dev_tensor)
    
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_acc:.4f}, "
          f"Dev Loss: {dev_loss.item():.4f}, Dev Acc: {dev_acc:.4f}")


torch.Size([79985, 11])
torch.Size([79985, 1])
Sequential(
  (0): Linear(in_features=11, out_features=5, bias=True)
  (1): ReLU()
  (2): Linear(in_features=5, out_features=1, bias=True)
)
Epoch 1/50, Train Loss: 0.6403, Dev Loss: 0.5288
Epoch 2/50, Train Loss: 0.4385, Dev Loss: 0.3800
Epoch 3/50, Train Loss: 0.3482, Dev Loss: 0.3379
Epoch 4/50, Train Loss: 0.3219, Dev Loss: 0.3220
Epoch 5/50, Train Loss: 0.3068, Dev Loss: 0.3081
Epoch 6/50, Train Loss: 0.2930, Dev Loss: 0.2948
Epoch 7/50, Train Loss: 0.2796, Dev Loss: 0.2818
Epoch 8/50, Train Loss: 0.2669, Dev Loss: 0.2696
Epoch 9/50, Train Loss: 0.2550, Dev Loss: 0.2579
Epoch 10/50, Train Loss: 0.2440, Dev Loss: 0.2471
Epoch 11/50, Train Loss: 0.2339, Dev Loss: 0.2373
Epoch 12/50, Train Loss: 0.2245, Dev Loss: 0.2279
Epoch 13/50, Train Loss: 0.2155, Dev Loss: 0.2184
Epoch 14/50, Train Loss: 0.2055, Dev Loss: 0.2075
Epoch 15/50, Train Loss: 0.1953, Dev Loss: 0.1973
Epoch 16/50, Train Loss: 0.1858, Dev Loss: 0.1878
Epoch 17/50, Train Lo

In [4]:
model.eval() 
y_test_tensor = y_test_tensor.view(-1,1).float()

with torch.no_grad():  # no gradients needed
    test_logits = model(X_test_tensor)  # raw logits
test_probs = torch.sigmoid(test_logits)  # convert logits to [0,1] probabilities
test_preds = (test_probs >= 0.5).int()  
accuracy = (test_preds == y_test_tensor.int()).float().mean()
print(f"Test Accuracy: {accuracy.item():.4f}")
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_true = y_test_tensor.numpy()
y_pred = test_preds.numpy()
y_prob = test_probs.numpy()

# Confusion matrix
print(confusion_matrix(y_true, y_pred))

# Classification report (precision, recall, F1)
print(classification_report(y_true, y_pred))

# ROC AUC
roc_auc = roc_auc_score(y_true, y_prob)
print(f"Test ROC AUC: {roc_auc:.4f}")

Test Accuracy: 0.9592
[[9010   87]
 [ 321  581]]
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      9097
         1.0       0.87      0.64      0.74       902

    accuracy                           0.96      9999
   macro avg       0.92      0.82      0.86      9999
weighted avg       0.96      0.96      0.96      9999

Test ROC AUC: 0.9640
